In [1]:
# Cell 1 — Setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

from src.features import FEATURE_COLS
from src.models import build_lgbm_baseline, build_xgb_baseline, build_catboost_baseline, build_ensemble_prediction
from src.models import save_model

df = pd.read_csv('../features/eurusd_labelled.csv', index_col=0, parse_dates=True)
X = df[FEATURE_COLS]
y = df['label_binary'].astype(int)

split_idx = int(len(X) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Train: 25,302 | Test: 6,326


In [2]:
# Cell 2 — Train ensemble and compare to single model
print("Training LGBM + XGBoost ensemble...")
ens_probs, lgbm_probs, xgb_probs, lgbm_model, xgb_model = build_ensemble_prediction(
    X_train, y_train, X_test
)

# Convert probs to binary predictions
threshold = 0.5
lgbm_preds = (lgbm_probs >= threshold).astype(int)
xgb_preds  = (xgb_probs  >= threshold).astype(int)
ens_preds  = (ens_probs   >= threshold).astype(int)

# High-confidence mask (ensemble)
hc_mask = (ens_probs >= 0.60) | (ens_probs <= 0.40)

print("\n" + "="*55)
print("  Model Comparison — Test Set")
print("="*55)
print(f"  LGBM alone         : {accuracy_score(y_test, lgbm_preds)*100:.2f}%")
print(f"  XGBoost alone      : {accuracy_score(y_test, xgb_preds)*100:.2f}%")
print(f"  Ensemble (avg)     : {accuracy_score(y_test, ens_preds)*100:.2f}%")
print()
print(f"  Ensemble HC signals: {hc_mask.sum()}")
if hc_mask.sum() >= 10:
    print(f"  Ensemble HC acc    : {accuracy_score(y_test[hc_mask], ens_preds[hc_mask])*100:.2f}%")
print(f"  Ensemble AUC-ROC   : {roc_auc_score(y_test, ens_probs):.4f}")
print("="*55)

Training LGBM + XGBoost ensemble...

  Model Comparison — Test Set
  LGBM alone         : 60.58%
  XGBoost alone      : 63.82%
  Ensemble (avg)     : 62.44%

  Ensemble HC signals: 1282
  Ensemble HC acc    : 79.95%
  Ensemble AUC-ROC   : 0.6384


In [3]:
# Cell 3 — Walk-forward ensemble validation
from src.validation import walk_forward_validate

def ensemble_model_builder():
    """
    Returns a wrapper that mimics the sklearn interface
    but internally trains both LGBM and XGBoost.
    """
    from sklearn.base import BaseEstimator, ClassifierMixin

    # class EnsembleClassifier(BaseEstimator, ClassifierMixin):
    #     def __init__(self):
    #         self.lgbm = build_lgbm_baseline()
    #         self.xgb  = build_xgb_baseline()

    #     def fit(self, X, y):
    #         self.lgbm.fit(X, y)
    #         self.xgb.fit(X, y)
    #         self.classes_ = np.array([0, 1])
    #         return self

    #     def predict_proba(self, X):
    #         lp = self.lgbm.predict_proba(X)[:, 1]
    #         xp = self.xgb.predict_proba(X)[:, 1]
    #         avg = 0.55 * lp + 0.45 * xp
    #         return np.column_stack([1 - avg, avg])

    #     def predict(self, X):
    #         return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    class EnsembleClassifier(BaseEstimator, ClassifierMixin):
        def __init__(self):
            self.lgbm = build_lgbm_baseline()
            self.xgb  = build_xgb_baseline()
            self.cat  = build_catboost_baseline()

        def fit(self, X, y):
            self.lgbm.fit(X, y)
            self.xgb.fit(X, y)
            self.cat.fit(X, y)
            self.classes_ = np.array([0, 1])
            return self

        def predict_proba(self, X):
            lp = self.lgbm.predict_proba(X)[:, 1]
            xp = self.xgb.predict_proba(X)[:, 1]
            cp = self.cat.predict_proba(X)[:, 1]
            # LGBM leads, XGB and CatBoost support
            avg = 0.45 * lp + 0.30 * xp + 0.25 * cp
            return np.column_stack([1 - avg, avg])

        def predict(self, X):
            return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    return EnsembleClassifier()

print("Running walk-forward validation on ensemble...")
print("(This will take 10-15 minutes)")

results_ens = walk_forward_validate(
    df            = df,
    feature_cols  = FEATURE_COLS,
    label_col     = 'label_binary',
    train_bars    = 2520,
    test_bars     = 630,
    step_bars     = 210,
    embargo_bars  = 24,
    confidence_thresh = 0.60,
    model_builder = ensemble_model_builder,
)

Running walk-forward validation on ensemble...
(This will take 10-15 minutes)
  Walk-forward validation
  Train: 2520 bars | Test: 630 bars | Step: 210 bars | Embargo: 24 bars
  Fold 01 | 2020-07-10 → 2020-08-26 | acc=0.606 | hc_acc=0.640 | n=630
  Fold 02 | 2020-07-27 → 2020-09-10 | acc=0.663 | hc_acc=0.713 | n=630
  Fold 03 | 2020-08-10 → 2020-09-24 | acc=0.692 | hc_acc=0.710 | n=630
  Fold 04 | 2020-08-26 → 2020-10-12 | acc=0.708 | hc_acc=0.738 | n=630
  Fold 05 | 2020-09-10 → 2020-10-27 | acc=0.702 | hc_acc=0.754 | n=630
  Fold 06 | 2020-09-24 → 2020-11-12 | acc=0.692 | hc_acc=0.722 | n=630
  Fold 07 | 2020-10-12 → 2020-11-27 | acc=0.656 | hc_acc=0.668 | n=630
  Fold 08 | 2020-10-27 → 2020-12-11 | acc=0.654 | hc_acc=0.662 | n=630
  Fold 09 | 2020-11-13 → 2020-12-28 | acc=0.608 | hc_acc=0.625 | n=630
  Fold 10 | 2020-11-27 → 2021-01-12 | acc=0.665 | hc_acc=0.745 | n=630
  Fold 11 | 2020-12-11 → 2021-01-28 | acc=0.659 | hc_acc=0.747 | n=630
  Fold 12 | 2020-12-28 → 2021-02-11 | acc=0

In [4]:
# Cell 4 — Compare LGBM-only vs Ensemble results side by side
lgbm_results = pd.read_csv('../results/walk_forward_results.csv')

print("\nComparison: LGBM alone vs Ensemble")
print("="*55)
print(f"  LGBM mean OOS acc  : {lgbm_results['accuracy'].mean()*100:.2f}%")
print(f"  Ensemble mean acc  : {results_ens['accuracy'].mean()*100:.2f}%")
print()
print(f"  LGBM mean HC acc   : {lgbm_results['hc_accuracy'].dropna().mean()*100:.2f}%")
print(f"  Ensemble HC acc    : {results_ens['hc_accuracy'].dropna().mean()*100:.2f}%")
print()
print(f"  LGBM worst fold    : {lgbm_results['accuracy'].min()*100:.2f}%")
print(f"  Ensemble worst     : {results_ens['accuracy'].min()*100:.2f}%")
print("="*55)

# Save ensemble results
results_ens.to_csv('../results/ensemble_walk_forward_results.csv', index=False)
print("\nSaved to results/ensemble_walk_forward_results.csv")


Comparison: LGBM alone vs Ensemble
  LGBM mean OOS acc  : 58.47%
  Ensemble mean acc  : 61.07%

  LGBM mean HC acc   : 68.16%
  Ensemble HC acc    : 72.39%

  LGBM worst fold    : 46.19%
  Ensemble worst     : 45.71%

Saved to results/ensemble_walk_forward_results.csv


In [5]:
from src.models import build_catboost_baseline

cat_model = build_catboost_baseline()
cat_model.fit(X_train, y_train)
print("[OK] CatBoost trained")


[OK] CatBoost trained


In [6]:
# Cell 5 — Save both models
save_model(lgbm_model, 'lgbm_v2.pkl')
save_model(xgb_model,  'xgb_v1.pkl')
save_model(cat_model,  'catboost_v1.pkl')
print("\nBoth models saved. You now have a 2-model ensemble.")
print("\nNext step: Add rate differential and CB stance features (Phase 2 macro integration).")

  [SAVED] Model saved to: C:\Users\omtej\Projects\2020_algo_based\models\lgbm_v2.pkl
  [SAVED] Model saved to: C:\Users\omtej\Projects\2020_algo_based\models\xgb_v1.pkl
  [SAVED] Model saved to: C:\Users\omtej\Projects\2020_algo_based\models\catboost_v1.pkl

Both models saved. You now have a 2-model ensemble.

Next step: Add rate differential and CB stance features (Phase 2 macro integration).
